In [ ]:
# 检查运行环境的库版本：不同版本的 PyTorch / tiktoken 在 API 行为上可能有差异，
# 先打印版本便于复现实验结果与排查兼容性问题。
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

In [ ]:
#Coding an LLM architecture

In [ ]:
# GPT_CONFIG_124M：GPT-2 small（约 1.24 亿参数）的超参数配置字典。
# 这些超参决定模型规模与行为，后续 GPTModel 及所有子模块都从此字典读取。
GPT_CONFIG_124M={
    # vocab_size：词表大小，等于 tokenizer 的 token 总数；决定 token 嵌入表与输出层的维度
    "vocab_size": 50257,    # Vocabulary size
    # context_length：最大上下文长度（一次能处理的 token 数上限），决定位置嵌入表的行数
    "context_length": 1024, # Context length
    # emb_dim：词向量/隐藏维度，贯穿整个模型的特征宽度
    "emb_dim": 768,         # Embedding dimension
    # n_heads：多头注意力的头数；要求 emb_dim 能被其整除（每头维度 = emb_dim/n_heads）
    "n_heads": 12,          # Number of attention heads
    # n_layers：堆叠的 TransformerBlock 数量，即模型深度
    "n_layers": 12,         # Number of layers
    # drop_rate：dropout 比例，0.0 表示不丢弃（便于确定性演示）
    "drop_rate": 0.0,       # Dropout rate
    # qkv_bias：Q/K/V 线性层是否加偏置项；GPT-2 原版为 False
    "qkv_bias": False       # Query-Key-Value bias
}

In [ ]:
# Coding the GPT model

In [ ]:
# ============================================================
# GPTModel：把所有构件组装成完整的 GPT。前向数据流：
#   token embedding + positional embedding 相加 -> dropout
#   -> 顺序通过 n_layers 个 TransformerBlock -> 最终 LayerNorm
#   -> out_head 线性投影到词表维度，得到每个位置对下一个 token 的 logits。
# ============================================================
import torch
import torch.nn as nn
from supplementary import TransformerBlock, LayerNorm
class GPTModel(nn.Module):
    # ✅ 已修复：原代码 `def __init__` 前为 5 个空格、`def forward` 为 4 个空格，
    #    缩进不一致会触发 IndentationError；现统一为 4 个空格对齐。
    def __init__(self, cfg):
        super().__init__()
        # tok_emb：token 嵌入表，形状 (vocab_size, emb_dim)，把 token id 映射为向量
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        # pos_emb：位置嵌入表，形状 (context_length, emb_dim)，为每个位置注入顺序信息
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        # drop_emb：嵌入相加后的 dropout 层
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # trf_blocks：用 nn.Sequential 顺序堆叠 n_layers 个 TransformerBlock（模型主体）
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        # final_norm：输出前的最终层归一化，稳定最后一层的激活分布
        self.final_norm = LayerNorm(cfg["emb_dim"])
        # out_head：无偏置线性层，把 emb_dim 投影到 vocab_size，得到 logits（对下一个 token 的打分）
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    # 前向传播：输入 token id (batch, seq_len) -> 输出 logits (batch, seq_len, vocab_size)
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        # 查 token 嵌入表：(batch, seq_len) -> (batch, seq_len, emb_dim)
        tok_embeds = self.tok_emb(in_idx)
        # 生成 0..seq_len-1 的位置索引并查位置嵌入表得到 (seq_len, emb_dim)，随后与 token 向量广播相加
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        # 相加后的嵌入做 dropout（正则化）
        x = self.drop_emb(x)
        # 依次通过所有 Transformer 层，形状保持 (batch, seq_len, emb_dim)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        # 投影到词表维度：(batch, seq_len, emb_dim) -> (batch, seq_len, vocab_size)
        logits = self.out_head(x)
        return logits

In [ ]:
# 构造一个演示用输入 batch：把两句话分别用 GPT-2 分词器编码成 token id，
# 再堆叠成形状 (2, num_tokens) 的张量（两句编码后 token 数需一致才能 stack）。
import torch
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

# encode 把字符串转成 token id 列表，torch.tensor 再转成张量
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
# stack 沿 dim=0 组成批次维度，得到形状 (batch=2, num_tokens)
batch = torch.stack(batch, dim=0)
print(batch)

In [ ]:
# 固定随机种子以复现权重初始化，实例化 GPTModel 并做一次前向传播。
# 此时权重是随机的，仅用于验证数据流与输出形状是否正确（尚未训练）。
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

# 前向传播，输出 logits 形状应为 (batch, num_tokens, vocab_size)
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

In [ ]:
#Generating text

In [ ]:
# ============================================================
# generate_text_simple：自回归（autoregressive）贪心解码，逐个生成新 token。
# 每一步流程：裁剪上下文 -> 前向得 logits -> 取最后一个位置 -> softmax
#           -> argmax 取概率最高的 token（贪心/greedy）-> 拼到序列末尾，
# 如此循环 max_new_tokens 次，把上一步的输出作为下一步的输入。
# ============================================================
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    # ✅ 已修复：原代码写成 `if _ in range(...)`，既不循环、`_` 又未定义（NameError）；
    #    现改为 `for _ in range(max_new_tokens):`，循环生成 max_new_tokens 个 token。
    for _ in range(max_new_tokens):
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        # ✅ 已修复：原代码 `idx[:,-context_size]` 缺少冒号，只取单个位置并降维；
        #    现改为 idx[:, -context_size:]，即取最后 context_size 个 token 作为上下文，
        #    以防序列超过模型支持的 context_length。
        idx_cond=idx[:, -context_size:]
          # Get the predictions
        # 推理不需要梯度，no_grad 可省显存、加速
        with torch.no_grad():
            logits = model(idx_cond)
            # Focus only on the last time step
            # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        # 只取序列最后一个位置的预测：(batch, n_tokens, vocab) -> (batch, vocab)
        logits=logits[:,-1,:]
        # Apply softmax to get probabilities
        # softmax 转成概率分布（贪心解码下 argmax 结果与直接对 logits 取 argmax 相同，此处为教学清晰）
        probas=torch.softmax(logits,dim=-1)
        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)
        # Append sampled index to the running sequence
        # 把新 token 追加到序列末尾，作为下一步的输入（体现自回归）
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)
    return idx

In [ ]:
#Exercise: Generate some text

In [ ]:
# 切换到评估模式：关闭 dropout 等训练专用行为，保证生成/推理结果确定。
model.eval()

In [ ]:
#Solution

In [ ]:
# 准备生成的起始提示（prompt）：编码成 token id 列表。
start_context = "Hello, I am"
encoded=tokenizer.encode(start_context)
print("encode:",encoded)
# unsqueeze(0) 在最前面加一个 batch 维，模型要求输入形状为 (batch, num_tokens)，得到 (1, n_tokens)
encoded_tensor=torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

In [ ]:
# 调用生成函数，从提示续写 6 个新 token。
# 注意：generate_text_simple 存在上述两处 bug，需修正后此单元才能正常运行。
out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

In [ ]:
# 把生成的 token id 序列解码回可读文本：squeeze(0) 去掉 batch 维，tolist 转普通列表后解码。
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)